# GIAI ĐOẠN 5b: Domain Eval cho 3 Model Ablation (không train, ~10-15 phút)## Vì sao notebook này tồn tạiNotebook 08 chạy xong 3 runs ablation + zip. Cell DOMAIN EVAL (cell 10 của 08)được xếp SÂU cell ZIP -> `ablation_domain_map50.csv` không được tạo.Session Kaggle mới không có `runs/detect/...` (bị xoá /kaggle/working).Nên notebook này:- Clone repo (lấy `data.loader` để dùng `YOLO_CLASS_NAMES`).- **KHÔNG cần attach dataset nào** - chỉ cần:  1. Chạy 1 cell 'SETUP' bên dưới (dùng `!unzip`).  2. Upload file `ablation_results.zip` (từ Kaggle Output của run 08)     và `al_results.zip` (từ Kaggle Output của run 07)     vào /kaggle/input qua tab Add Dataset / Attach.## 2 file cần attach (Kaggle Datasets -> New Dataset)| File upload | Tên dataset | Trỏ tới trong code ||---|---|---|| ablation_results.zip | `ablation_results` | `/kaggle/input/*/ablation_results/` (3 CSV, 3 checkpoint, 3 best.pt, eval_set, _shared_eval_metadata.csv) || al_results.zip | `al_results` | `/kaggle/input/*/al_results/` (CSV + best.pt của main experiment, 6 runs, 3 seeds) |Nếu không upload được (quota, v.v.), chạy cell 1 + cell 3 với`ABL_DIR = ""` -> sẽ chỉ làm được gì mình có sẵn trong zip.## Kết quả mong đợi1 file CSV: `ablation_domain_map50.csv` + 1 bảng in ra:  4 domain (night / adverse_weather / daytime_ctrl / rare_scenes)  x 5 model (baseline_seed42, sqrt_main, abl_inverse, abl_equal, abl_proportional)Mỗi ô = mAP50 trên fixed eval set 2000 ảnh (seed 2024, cùng với main exp).Sau khi in bảng, copy kết quả -> gửi Claude tổng hợp.

In [ ]:
## CELL 1: ENV + CLONE REPO (KHÔNG CẦN ATTACH Datasets)import os as _os_os.environ["OMP_NUM_THREADS"] = "1"_os.environ["MKL_NUM_THREADS"] = "1"!pip install ultralytics==8.4.184 torch scikit-learn pandas pyyaml -q 2>/dev/null || !pip install ultralytics==8.4.184 torch -q!rm -rf /kaggle/working/metadata-stratified-active-learning!git clone -b develop https://github.com/nguyendinhtanloc-Org/metadata-stratified-active-learning.gitimport syssys.path.insert(0, "/kaggle/working/metadata-stratified-active-learning/src")print("[OK] repo + deps ready")

In [ ]:
## CELL 2: Tìm dataset đã giải nén hoặc ZIP đầu vào
from pathlib import Path
import glob, os

DATA_ROOT = Path('/kaggle/input/datasets/nguyendinhtanloc1')
ABL_DATA = DATA_ROOT / 'ablation-results'
AL_DATA = DATA_ROOT / 'al-results' / 'al_results'

# Notebook hỗ trợ cả cấu trúc dataset đã giải nén và ZIP.
def find_nested(root, patterns):
    for pattern in patterns:
        found = sorted(root.rglob(pattern))
        if found:
            return found[0]
    return None

ABL_ZIP = find_nested(DATA_ROOT, ['ablation_results.zip', 'ablation-results.zip'])
MAIN_ZIP = find_nested(DATA_ROOT, ['al_results.zip', 'al-results.zip'])

print('ABL_DATA:', ABL_DATA if ABL_DATA.exists() else 'not found')
print('AL_DATA :', AL_DATA if AL_DATA.exists() else 'not found')
print('ABL_ZIP :', ABL_ZIP)
print('MAIN_ZIP:', MAIN_ZIP)

if not ABL_DATA.exists() and ABL_ZIP is None:
    raise FileNotFoundError('Không tìm thấy ablation-results hoặc ablation_results.zip')
if not AL_DATA.exists() and MAIN_ZIP is None:
    raise FileNotFoundError('Không tìm thấy al-results/al_results hoặc al_results.zip')

In [ ]:
## CELL 3: Chuẩn hóa thư mục input, eval set và model weights
import os, shutil, zipfile

ABL_DIR = '/kaggle/working/abl_extracted'
MAIN_DIR = '/kaggle/working/main_extracted'
for d in (ABL_DIR, MAIN_DIR):
    if os.path.isdir(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

def unzip(src, dst):
    with zipfile.ZipFile(src) as z:
        z.extractall(dst)

# Dataset đã giải nén: dùng trực tiếp. ZIP: giải nén vào working.
if ABL_DATA.exists():
    ABL_DIR = str(ABL_DATA)
elif ABL_ZIP is not None:
    unzip(str(ABL_ZIP), ABL_DIR)

if AL_DATA.exists():
    MAIN_DIR = str(AL_DATA)
elif MAIN_ZIP is not None:
    unzip(str(MAIN_ZIP), MAIN_DIR)

# Tìm file metadata/eval/model đệ quy để xử lý thư mục lồng nhau.
def find_file(root, pattern):
    hits = sorted(Path(root).rglob(pattern))
    return str(hits[0]) if hits else None

def find_dir(root, name):
    hits = sorted(p for p in Path(root).rglob(name) if p.is_dir())
    return str(hits[0]) if hits else None

EVAL_META_CSV = find_file(ABL_DIR, '_shared_eval_metadata.csv') or find_file(MAIN_DIR, '_shared_eval_metadata.csv')
assert EVAL_META_CSV, 'Không thấy _shared_eval_metadata.csv'

SHARED_EVAL_DIR = find_dir(MAIN_DIR, 'eval_set') or find_dir(ABL_DIR, 'eval_set')
assert SHARED_EVAL_DIR, 'Không thấy eval_set'

ABL_BESTPT = {
    'abl_inverse_seed42': find_file(ABL_DIR, 'abl_inverse_seed42/best.pt'),
    'abl_equal_seed42': find_file(ABL_DIR, 'abl_equal_seed42/best.pt'),
    'abl_proportional_seed42': find_file(ABL_DIR, 'abl_proportional_seed42/best.pt'),
    'sqrt_main_seed42': find_file(MAIN_DIR, 'stratified_seed42/best.pt'),
    'baseline_seed42': find_file(MAIN_DIR, 'baseline_seed42/best.pt'),
}
print('ABL_DIR:', ABL_DIR)
print('MAIN_DIR:', MAIN_DIR)
print('EVAL_META_CSV:', EVAL_META_CSV)
print('SHARED_EVAL_DIR:', SHARED_EVAL_DIR)
for name, path in ABL_BESTPT.items():
    print(f'{name}:', path if path and os.path.exists(path) else 'MISSING')

In [ ]:
## CELL 4: Dựng domain sub-eval sets (hardlink từ SHARED_EVAL_DIR, 4 domain)import os, glob, shutil, yamlimport pandas as pdfrom data.loader import YOLO_CLASS_NAMESDOMAIN_ROOT = "/kaggle/working/data/processed/_domain_eval_abl2"os.makedirs(DOMAIN_ROOT, exist_ok=True)meta = pd.read_csv(EVAL_META_CSV)vc = meta["scene"].value_counts()rare_scenes = set(vc[vc < 0.3 * vc.max()].index) - {"undefined", "unknown", "street"}print("rare_scenes threshold-30% ->", sorted(rare_scenes))DOMAINS = {    "night":           lambda m: m["timeofday"] == "night",    "adverse_weather": lambda m: m["weather"] in ("rainy", "snowy", "foggy"),    "daytime_ctrl":    lambda m: m["timeofday"] == "daytime",    "rare_scenes":     lambda m: m["scene"] in rare_scenes,}for dname, pred in DOMAINS.items():    sel = meta[meta.apply(lambda r: pred(r), axis=1)]    if sel.empty:        print(f"{dname}: 0 anh -> skip")        continue    ddir = os.path.join(DOMAIN_ROOT, dname)    for sub in ("images", "labels"):        os.makedirs(os.path.join(ddir, sub), exist_ok=True)    n = 0    for _, row in sel.iterrows():        fname = os.path.basename(str(row["filename"]))        stem = os.path.splitext(fname)[0]        for s, d in [(f"{SHARED_EVAL_DIR}/images/{fname}", f"{ddir}/images/{fname}"),                     (f"{SHARED_EVAL_DIR}/labels/{stem}.txt", f"{ddir}/labels/{stem}.txt")]:            if os.path.exists(s) and not os.path.exists(d):                try: os.link(s, d); n += 1                except OSError: shutil.copy2(s, d); n += 1    ycfg = {"path": os.path.abspath(ddir), "train": "images", "val": "images",            "names": list(YOLO_CLASS_NAMES.values()) if isinstance(YOLO_CLASS_NAMES, dict) else list(YOLO_CLASS_NAMES)}    with open(f"{ddir}/domain.yaml", "w") as f: yaml.dump(ycfg, f)    print(f"[OK] {dname}: {len(sel)} anh metadata / {n} file hardlinked -> {ddir}/domain.yaml")

In [ ]:
## CELL 5: Đánh giá 5 model x 4 domain, lưu đầy đủ mAP50 và mAP50-95, ZIP output
import os
import zipfile
from pathlib import Path
import pandas as pd
import numpy as np
import torch
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else 'cpu'
print('Val device:', device)
rows = []

for exp, wpt in ABL_BESTPT.items():
    if not wpt or not os.path.exists(wpt):
        print(f'[SKIP] {exp}: thiếu best.pt')
        continue
    print(f'\n[MODEL] {exp}')
    model = YOLO(wpt)
    for dname in DOMAINS:
        dyaml = f'{DOMAIN_ROOT}/{dname}/domain.yaml'
        if not os.path.exists(dyaml):
            print(f'  [SKIP] {dname}: thiếu domain.yaml')
            continue
        res = model.val(data=dyaml, device=device, plots=False, verbose=False)
        rows.append({
            'experiment': exp,
            'domain': dname,
            'map50': round(float(res.box.map50), 4),
            'map50_95': round(float(res.box.map), 4),
            'n_images': sum(1 for _ in Path(f'{DOMAIN_ROOT}/{dname}/images').glob('*')),
        })
        print(f"  {dname:15s} mAP50={rows[-1]['map50']:.4f} mAP50-95={rows[-1]['map50_95']:.4f}")
    del model

out_tab = pd.DataFrame(rows)
if out_tab.empty:
    raise RuntimeError('Không có kết quả đánh giá nào được tạo')

order = [c for c in ['baseline_seed42', 'abl_proportional_seed42',
                     'abl_equal_seed42', 'abl_inverse_seed42',
                     'sqrt_main_seed42'] if c in out_tab['experiment'].unique()]
piv50 = out_tab.pivot(index='domain', columns='experiment', values='map50').reindex(columns=order).round(4)
piv95 = out_tab.pivot(index='domain', columns='experiment', values='map50_95').reindex(columns=order).round(4)

OUT_DIR = Path('/kaggle/working/output')
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_tab.to_csv(OUT_DIR / 'ablation_domain_eval_long.csv', index=False)
piv50.to_csv(OUT_DIR / 'ablation_domain_map50.csv')
piv95.to_csv(OUT_DIR / 'ablation_domain_map50_95.csv')
summary = out_tab.groupby('experiment').agg(
    domains_evaluated=('domain', 'nunique'),
    mean_map50=('map50', 'mean'),
    mean_map50_95=('map50_95', 'mean'),
    min_map50=('map50', 'min'),
    max_map50=('map50', 'max'),
).reset_index()
summary.to_csv(OUT_DIR / 'ablation_domain_eval_summary.csv', index=False)

# ZIP gọn 4 file CSV để tải về dễ.
zip_path = Path('/kaggle/working/ablation_domain_output.zip')
with zipfile.ZipFile(zip_path, 'w') as zf:
    for f in sorted(OUT_DIR.glob('*.csv')):
        zf.write(f, arcname=f.name)

print('\nDOMAIN mAP50')
print(piv50.to_string())
print('\nDOMAIN mAP50-95')
print(piv95.to_string())
print('\n[SAVED] 4 file CSV trong', OUT_DIR)
print('[SAVED] ZIP:', zip_path, '(', zip_path.stat().st_size, 'bytes )')
print('>>> Download file ablation_domain_output.zip từ tab Output của Kaggle')